LIME explanation

In [ ]:

try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries

    # classifier fn for explainers: returns [p(NORMAL), p(PNEUMONIA)]
    def predict_fn_for_explainers(images):
        imgs = []
        for img in images:
            arr = img.astype(np.uint8)
            resized = cv2.resize(arr, (IMG_SIZE, IMG_SIZE))
            pre = densenet_preprocess(resized.astype(np.float32))
            imgs.append(pre)
        batch = np.stack(imgs, axis=0)
        probs_pos = model.predict(batch).ravel()
        probs_neg = 1.0 - probs_pos
        return np.vstack([probs_neg, probs_pos]).T

    print("Running LIME (this may take some time)...")
    explainer = lime_image.LimeImageExplainer(random_state=42)

    explanation = explainer.explain_instance(
        image=orig_rgb_resized,
        classifier_fn=predict_fn_for_explainers,
        top_labels=2,
        hide_color=0,
        num_samples=300,
        batch_size=50
    )

    # explain pneumonia class (label index = 1)
    label_to_explain = 1
    temp, mask = explanation.get_image_and_mask(
        label=label_to_explain,
        positive_only=False,
        num_features=10,
        hide_rest=False
    )

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.title("Original"); plt.imshow(orig_rgb_resized); plt.axis('off')
    plt.subplot(1,2,2); plt.title("LIME (important regions)");
    plt.imshow(mark_boundaries(temp/255.0, mask)); plt.axis('off')
    plt.tight_layout()
    plt.show()

    print("LIME explanation done.")
except Exception as e:
    print("LIME failed or not installed. Error:", e)
